# Title

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline
import pandas

pandas.options.display.max_rows = 6

In [2]:
from tsdm.datasets import BaseDataset

In [3]:
class USHCN(BaseDataset):
    url = "https://cdiac.ess-dive.lbl.gov/ftp/ushcn_daily/"

In [4]:
# USHCN.download()

# State Codes

In [5]:
# Best viewed with elastic tabstops!
state_codes = """
State	Postal Abbr.	FIPS code
Alabama	AL	01
Alaska	AK	02
Arizona	AZ	04
Arkansas	AR	05
California	CA	06
Colorado	CO	08
Connecticut	CT	09
Delaware	DE	10
District of Columbia	DC	11
Florida	FL	12
Georgia	GA	13
Hawaii	HI	15
Idaho	ID	16
Illinois	IL	17
Indiana	IN	18
Iowa	IA	19
Kansas	KS	20
Kentucky	KY	21
Louisiana	LA	22
Maine	ME	23
Maryland	MD	24
Massachusetts	MA	25
Michigan	MI	26
Minnesota	MN	27
Mississippi	MS	28
Missouri	MO	29
Montana	MT	30
Nebraska	NE	31
Nevada	NV	32
New Hampshire	NH	33
New Jersey	NJ	34
New Mexico	NM	35
New York	NY	36
North Carolina	NC	37
North Dakota	ND	38
Ohio	OH	39
Oklahoma	OK	40
Oregon	OR	41
Pennsylvania	PA	42
Puerto Rico	PR	72
Rhode Island	RI	44
South Carolina	SC	45
South Dakota	SD	46
Tennessee	TN	47
Texas	TX	48
Utah	UT	49
Vermont	VT	50
Virginia	VA	51
Virgin Islands	VI	78
Washington	WA	53
West Virginia	WV	54
Wisconsin	WI	55
Wyoming	WY	56
"""

state_dtypes = {
    "State": pandas.StringDtype(),
    "Postal Abbr.": pandas.CategoricalDtype(ordered=True),
    "FIPS code": pandas.CategoricalDtype(ordered=True),
}

{'State': string[python],
 'Postal Abbr.': CategoricalDtype(categories=None, ordered=True),
 'FIPS code': CategoricalDtype(categories=None, ordered=True)}

In [6]:
from io import StringIO

states = pandas.read_csv(
    StringIO(state_codes), sep="\t", dtype=state_dtypes, index_col="Postal Abbr."
)

,State,FIPS code
Postal Abbr.,,
AL,Alabama,01
AK,Alaska,02
AZ,Arizona,04
...,...,...
WV,West Virginia,54
WI,Wisconsin,55
WY,Wyoming,56


# Stations Meta-Data

In [7]:
station_colspecs = {
    "COOP_ID": (1, 6),
    "LATITUDE": (8, 15),
    "LONGITUDE": (17, 25),
    "ELEVATION": (27, 32),
    "STATE": (34, 35),
    "NAME": (37, 66),
    "COMPONENT_1": (68, 73),
    "COMPONENT_2": (75, 80),
    "COMPONENT_3": (82, 87),
    "UTC_OFFSET": (89, 90),
}

# fix colspec to 0-index, half open interval
station_colspecs = {key: (a - 1, b) for key, (a, b) in station_colspecs.items()}

station_dtypes = {
    "COOP_ID": pandas.CategoricalDtype(ordered=True),
    "LATITUDE": pandas.Float32Dtype(),
    "LONGITUDE": pandas.Float32Dtype(),
    "ELEVATION": pandas.Float32Dtype(),
    "STATE": states.index.dtype,
    "NAME": pandas.StringDtype(),
    "COMPONENT_1": pandas.CategoricalDtype(ordered=True),
    "COMPONENT_2": pandas.CategoricalDtype(ordered=True),
    "COMPONENT_3": pandas.CategoricalDtype(ordered=True),
    "UTC_OFFSET": "timedelta64[h]",
}

station_na_values = {
    "ELEVATION": -999.9,
    "COMPONENT_1": "------",
    "COMPONENT_2": "------",
    "COMPONENT_3": "------",
}

{'ELEVATION': -999.9,
 'COMPONENT_1': '------',
 'COMPONENT_2': '------',
 'COMPONENT_3': '------'}

In [8]:
stations_filename = "ushcn-stations.txt"
stations_filepath = USHCN.rawdata_path.joinpath(stations_filename)
stations = pandas.read_fwf(
    stations_filepath,
    na_values=station_na_values,
    colspecs=list(station_colspecs.values()),
    header=0,
    names=station_colspecs,
    dtype=station_dtypes,
)
COOP_IDS = pandas.CategoricalDtype(stations.COOP_ID, ordered=True)
stations.astype(
    {
        "COOP_ID": COOP_IDS,
        "COMPONENT_1": COOP_IDS,
        "COMPONENT_2": COOP_IDS,
        "COMPONENT_3": COOP_IDS,
    }
)

,COOP_ID,LATITUDE,LONGITUDE,ELEVATION,STATE,NAME,COMPONENT_1,COMPONENT_2,COMPONENT_3,UTC_OFFSET
0,012813,30.5467,-87.880798,7.0,AL,FAIRHOPE 2 NE,NaN,NaN,NaN,0 days 06:00:00
1,013160,32.834702,-88.134201,38.099998,AL,GAINESVILLE LOCK,NaN,NaN,NaN,0 days 06:00:00
2,013511,32.701698,-87.580803,67.099998,AL,GREENSBORO,NaN,NaN,NaN,0 days 06:00:00
...,...,...,...,...,...,...,...,...,...,...
1214,489615,42.1106,-104.949203,1413.699951,WY,WHEATLAND 4 N,NaN,NaN,NaN,0 days 07:00:00
1215,489770,44.010799,-107.968597,1237.5,WY,WORLAND,NaN,NaN,NaN,0 days 07:00:00
1216,489905,44.9767,-110.696404,1898.900024,WY,YELLOWSTONE PK MAMMOTH,NaN,NaN,NaN,0 days 07:00:00


# Station Data

In [9]:
MFLAGS = pandas.CategoricalDtype(
    categories=("B", "D", "H", "K", "L", "O", "P", "T", "W")
)
QFLAGS = pandas.CategoricalDtype(
    categories=("D", "G", "I", "K", "L", "M", "N", "O", "R", "S", "T", "W", "X", "Z")
)
SFLAGS = pandas.CategoricalDtype(
    categories=(
        "0",
        "6",
        "7",
        "A",
        "B",
        "F",
        "G",
        "H",
        "K",
        "M",
        "N",
        "R",
        "S",
        "T",
        "U",
        "W",
        "X",
        "Z",
    )
)
ELEMENTS = pandas.CategoricalDtype(categories=("PRCP", "SNOW", "SNWD", "TMAX", "TMIN"))


dtypes = {
    "COOP_ID": COOP_IDS,
    "YEAR": pandas.Int16Dtype(),
    "MONTH": pandas.Int16Dtype(),
    "ELEMENT": ELEMENTS,
    "VALUE": pandas.Int16Dtype(),
    "MFLAG": MFLAGS,
    "QFLAG": QFLAGS,
    "SFLAG": SFLAGS,
}

# column start, stop, dtype
colspecs = {
    "COOP_ID": (1, 6),
    "YEAR": (7, 10),
    "MONTH": (11, 12),
    "ELEMENT": (13, 16),
}

for k, i in enumerate(range(17, 258, 8)):
    colspecs |= {
        ("VALUE", k + 1): (i, i + 4),
        ("MFLAG", k + 1): (i + 5, i + 5),
        ("QFLAG", k + 1): (i + 6, i + 6),
        ("SFLAG", k + 1): (i + 7, i + 7),
    }

    # dtype |= {
    #     f"VALUE-{k+1}" : integer,
    #     f"MFLAG-{k+1}" : mflag_types,
    #     f"QFLAG-{k+1}" : qflag_types,
    #     f"SFLAG-{k+1}" : sflag_types,
    # }


# These should coincide with the description in data_format.txt
widths = [b - a + 1 for a, b in colspecs.values()]
dtype = {
    key: (dtypes[key[0]] if isinstance(key, tuple) else dtypes[key]) for key in colspecs
}

cspec = [(a - 1, b - 1) for a, b in colspecs.values()]
# na_values = [-9999]
# ds = pandas.read_fwf("state32.txt", names=colspecs, widths=widths, header=None, dtype=dtype, na_values=-9999)

[(0, 5),
 (6, 9),
 (10, 11),
 (12, 15),
 (16, 20),
 (21, 21),
 (22, 22),
 (23, 23),
 (24, 28),
 (29, 29),
 (30, 30),
 (31, 31),
 (32, 36),
 (37, 37),
 (38, 38),
 (39, 39),
 (40, 44),
 (45, 45),
 (46, 46),
 (47, 47),
 (48, 52),
 (53, 53),
 (54, 54),
 (55, 55),
 (56, 60),
 (61, 61),
 (62, 62),
 (63, 63),
 (64, 68),
 (69, 69),
 (70, 70),
 (71, 71),
 (72, 76),
 (77, 77),
 (78, 78),
 (79, 79),
 (80, 84),
 (85, 85),
 (86, 86),
 (87, 87),
 (88, 92),
 (93, 93),
 (94, 94),
 (95, 95),
 (96, 100),
 (101, 101),
 (102, 102),
 (103, 103),
 (104, 108),
 (109, 109),
 (110, 110),
 (111, 111),
 (112, 116),
 (117, 117),
 (118, 118),
 (119, 119),
 (120, 124),
 (125, 125),
 (126, 126),
 (127, 127),
 (128, 132),
 (133, 133),
 (134, 134),
 (135, 135),
 (136, 140),
 (141, 141),
 (142, 142),
 (143, 143),
 (144, 148),
 (149, 149),
 (150, 150),
 (151, 151),
 (152, 156),
 (157, 157),
 (158, 158),
 (159, 159),
 (160, 164),
 (165, 165),
 (166, 166),
 (167, 167),
 (168, 172),
 (173, 173),
 (174, 174),
 (175, 175),
 

In [10]:
from zipfile import ZipFile
import gzip

fname = "state01_AL.txt"
fpath = USHCN.rawdata_path.joinpath(fname + ".gz")

PosixPath('/home/rscholz/.tsdm/rawdata/USHCN/state01_AL.txt.gz')

In [62]:
%%time
with gzip.open(fpath) as file:
    ds = pandas.read_fwf(
        file, names=colspecs, widths=widths, header=None, dtype=dtype, na_values=-9999
    )

ds

CPU times: user 3.64 s, sys: 75.6 ms, total: 3.71 s
Wall time: 3.69 s


,COOP_ID,YEAR,MONTH,ELEMENT,"(VALUE, 1)","(MFLAG, 1)","(QFLAG, 1)","(SFLAG, 1)","(VALUE, 2)","(MFLAG, 2)",...,"(QFLAG, 29)","(SFLAG, 29)","(VALUE, 30)","(MFLAG, 30)","(QFLAG, 30)","(SFLAG, 30)","(VALUE, 31)","(MFLAG, 31)","(QFLAG, 31)","(SFLAG, 31)"
0,NaN,1926,1,TMAX,<NA>,NaN,NaN,NaN,<NA>,NaN,...,NaN,6,55,NaN,NaN,6,68,NaN,NaN,6
1,NaN,1926,1,TMIN,<NA>,NaN,NaN,NaN,<NA>,NaN,...,NaN,6,41,NaN,NaN,6,44,NaN,NaN,6
2,NaN,1926,1,SNOW,0,NaN,NaN,6,0,NaN,...,NaN,6,0,NaN,NaN,6,0,NaN,NaN,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81899,018469,2014,12,PRCP,0,NaN,NaN,7,0,NaN,...,NaN,7,6,NaN,NaN,7,<NA>,NaN,NaN,NaN
81900,018469,2014,12,SNOW,0,NaN,NaN,7,0,NaN,...,NaN,7,0,NaN,NaN,7,<NA>,NaN,NaN,NaN
81901,018469,2014,12,SNWD,0,NaN,NaN,7,0,NaN,...,NaN,7,0,NaN,NaN,7,<NA>,NaN,NaN,NaN


# preprocessing the data

In [17]:
id_cols = ["COOP_ID", "YEAR", "MONTH", "ELEMENT"]
data_cols = ["VALUE", "MFLAG", "QFLAG", "SFLAG"]
data_cols = [col for col in ds.columns if col not in id_cols]
columns = pandas.MultiIndex.from_tuples(ds[data_cols], names=["VAR", "DAY"])
data = pandas.DataFrame(ds[data_cols], columns=columns)
data.index.name = "INDEX"
data

VAR,VALUE,MFLAG,QFLAG,SFLAG,VALUE,MFLAG,QFLAG,SFLAG,VALUE,MFLAG,...,QFLAG,SFLAG,VALUE,MFLAG,QFLAG,SFLAG,VALUE,MFLAG,QFLAG,SFLAG
DAY,1,1,1,1,2,2,2,2,3,3,...,29,29,30,30,30,30,31,31,31,31
INDEX,,,,,,,,,,,,,,,,,,,,,
0,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,...,NaN,6,55,NaN,NaN,6,68,NaN,NaN,6
1,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,...,NaN,6,41,NaN,NaN,6,44,NaN,NaN,6
2,0,NaN,NaN,6,0,NaN,NaN,6,0,NaN,...,NaN,6,0,NaN,NaN,6,0,NaN,NaN,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6728236,0,NaN,NaN,7,0,NaN,NaN,7,0,NaN,...,NaN,7,0,T,NaN,7,0,NaN,NaN,7
6728237,0,NaN,NaN,7,0,NaN,NaN,7,0,NaN,...,NaN,7,0,T,NaN,7,0,NaN,NaN,7
6728238,1,NaN,NaN,7,1,NaN,NaN,7,1,NaN,...,NaN,7,7,NaN,NaN,7,7,NaN,NaN,7


In [18]:
%%time
# Pure magic https://stackoverflow.com/a/27044843/9318372
data = data.stack(level="DAY", dropna=False).reset_index(level="DAY")

VAR,DAY,MFLAG,QFLAG,SFLAG,VALUE
INDEX,,,,,
0,1,NaN,NaN,NaN,NaN
0,2,NaN,NaN,NaN,NaN
0,3,NaN,NaN,NaN,NaN
...,...,...,...,...,...
6728238,29,NaN,NaN,7,7
6728238,30,NaN,NaN,7,7
6728238,31,NaN,NaN,7,7


In [ ]:
%%time
data = ds[id_cols].join(data, how="inner").reset_index()
data = data.astype(dtypes | {"DAY": integer})
data = data[
    ["COOP_ID", "YEAR", "MONTH", "DAY", "ELEMENT", "MFLAG", "QFLAG", "SFLAG", "VALUE"]
]

In [ ]:
%%time
mask = pandas.isna(data[["MFLAG", "QFLAG", "SFLAG", "VALUE"]]).sum(axis=1) < 4
data = data[mask]
data = data.sort_values(by=["YEAR", "MONTH", "DAY", "COOP_ID", "ELEMENT"]).reset_index(
    drop=True
)
data

# ALternative: Use Modin for speedup

In [10]:
rayimport os
import ray
ray.init()

os.environ["MODIN_ENGINE"] = "ray"  # Modin will use Ray
# os.environ["MODIN_ENGINE"] = "dask"  # Modin will use Dask

2021-10-02 18:19:33,304	INFO services.py:1263 -- View the Ray dashboard at http://127.0.0.1:8265


In [13]:
# problem: currently only works uncompressed.

from modin import pandas as pd

fname = "us.txt"
fpath2 = USHCN.rawdata_path.joinpath(fname)

PosixPath('/home/rscholz/.tsdm/rawdata/USHCN/us.txt')

In [14]:
%%time
ds = pd.read_fwf(
    fpath2, names=colspecs, widths=widths, header=None, na_values=-9999, dtype=dtype
)

CPU times: user 1.14 s, sys: 682 ms, total: 1.82 s
Wall time: 27.4 s


In [29]:
id_cols = ["COOP_ID", "YEAR", "MONTH", "ELEMENT"]
data_cols = ["VALUE", "MFLAG", "QFLAG", "SFLAG"]
data_cols = [col for col in ds.columns if col not in id_cols]
columns = pd.MultiIndex.from_tuples(ds[data_cols], names=["VAR", "DAY"])
data = pd.DataFrame(ds[data_cols], columns=columns)
# data.columns  = columns
# data.index.name="INDEX"
data

,"(VALUE, 1)","(MFLAG, 1)","(QFLAG, 1)","(SFLAG, 1)","(VALUE, 2)","(MFLAG, 2)","(QFLAG, 2)","(SFLAG, 2)","(VALUE, 3)","(MFLAG, 3)",...,"(QFLAG, 29)","(SFLAG, 29)","(VALUE, 30)","(MFLAG, 30)","(QFLAG, 30)","(SFLAG, 30)","(VALUE, 31)","(MFLAG, 31)","(QFLAG, 31)","(SFLAG, 31)"
0,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,...,NaN,6,55,NaN,NaN,6,68,NaN,NaN,6
1,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,...,NaN,6,41,NaN,NaN,6,44,NaN,NaN,6
2,0,NaN,NaN,6,0,NaN,NaN,6,0,NaN,...,NaN,6,0,NaN,NaN,6,0,NaN,NaN,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6728236,0,NaN,NaN,7,0,NaN,NaN,7,0,NaN,...,NaN,7,0,T,NaN,7,0,NaN,NaN,7
6728237,0,NaN,NaN,7,0,NaN,NaN,7,0,NaN,...,NaN,7,0,T,NaN,7,0,NaN,NaN,7
6728238,1,NaN,NaN,7,1,NaN,NaN,7,1,NaN,...,NaN,7,7,NaN,NaN,7,7,NaN,NaN,7


MultiIndex([(0, 'a'),
            (1, 'b'),
            (2, 'c')],
           names=['NUMBER', 'LETTER'])

In [25]:
pd.DataFrame(data)

VAR,VALUE,MFLAG,QFLAG,SFLAG,VALUE,MFLAG,QFLAG,SFLAG,VALUE,MFLAG,...,QFLAG,SFLAG,VALUE,MFLAG,QFLAG,SFLAG,VALUE,MFLAG,QFLAG,SFLAG
DAY,1,1,1,1,2,2,2,2,3,3,...,29,29,30,30,30,30,31,31,31,31
0,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,...,NaN,6,55,NaN,NaN,6,68,NaN,NaN,6
1,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,...,NaN,6,41,NaN,NaN,6,44,NaN,NaN,6
2,0,NaN,NaN,6,0,NaN,NaN,6,0,NaN,...,NaN,6,0,NaN,NaN,6,0,NaN,NaN,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6728236,0,NaN,NaN,7,0,NaN,NaN,7,0,NaN,...,NaN,7,0,T,NaN,7,0,NaN,NaN,7
6728237,0,NaN,NaN,7,0,NaN,NaN,7,0,NaN,...,NaN,7,0,T,NaN,7,0,NaN,NaN,7
6728238,1,NaN,NaN,7,1,NaN,NaN,7,1,NaN,...,NaN,7,7,NaN,NaN,7,7,NaN,NaN,7


In [16]:
%%time
# Pure magic https://stackoverflow.com/a/27044843/9318372
data = data.stack(level="DAY", dropna=False).reset_index(level="DAY")

CPU times: user 16.6 s, sys: 9.7 s, total: 26.3 s
Wall time: 26.2 s


In [17]:
%%time
data = ds[id_cols]._to_pandas().join(data, how="inner").reset_index()
data = data.astype(dtypes | {"DAY": integer})
data = data[
    ["COOP_ID", "YEAR", "MONTH", "DAY", "ELEMENT", "MFLAG", "QFLAG", "SFLAG", "VALUE"]
]

AttributeError: 'str' object has no attribute 'columns'

In [ ]:
%%time
mask = pandas.isna(data[["MFLAG", "QFLAG", "SFLAG", "VALUE"]]).sum(axis=1) < 4
data = data[mask]
data = data.sort_values(by=["YEAR", "MONTH", "DAY", "COOP_ID", "ELEMENT"]).reset_index(
    drop=True
)
data